#### Load Data

In [1]:
import re
from langchain_community.document_loaders import Docx2txtLoader

In [2]:
docx_loader = Docx2txtLoader('../../data/Introduction_to_Data_and_Data_Science.docx')
docs = docx_loader.load()

for doc in docs:
    doc.page_content = re.sub('\n\n', '<<Para>>', doc.page_content)

    doc.page_content = re.sub('\n', ' ', doc.page_content)

    doc.page_content = re.sub('<<Para>>', '\n\n', doc.page_content)

print(len(docs))

1


#### Split Data

> split docs into a chunk to fit into the model's context limit

> model tends to perform better with each chunk covering a single topic

##### Character Text Splitter

> drawback: we can't ensure that each chunk contains a single topic

In [3]:
from langchain_text_splitters import CharacterTextSplitter

> chunk overlap allows for a smoother flow between chunks

> separator ensures the chunk doesn't end mid-word or mid-sentence, however, that separator is removed from text when separating

> using overlap with a custom separator like `.` starts the next chunk with previous sentence only if the sentence falls under the overlap length

In [4]:
char_splitter = CharacterTextSplitter(separator = ".", chunk_size = 500, chunk_overlap = 100)

chunks = char_splitter.split_documents(docs)

print(len(chunks))

22


In [5]:
print(chunks[1].page_content[-150:])
print(chunks[2].page_content[:150])

you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell
And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the 


##### Markdown Header Text Splitter

> loaded data must have `#` symbols for mardown headings

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [7]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [("#", "Document Title"), ("##", "Topic Title")])

for doc in docs:
    md_chunks = md_splitter.split_text(doc.page_content)

In [8]:
print(len(md_chunks))
print(md_chunks[0].metadata)
print(md_chunks[1].metadata)

2
{'Document Title': 'Introduction to Data and Data Science', 'Topic Title': 'Analysis vs Analytics'}
{'Document Title': 'Introduction to Data and Data Science', 'Topic Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}


In [9]:
# remove '\n' chars
for chunk in md_chunks:
    chunk.page_content = re.sub('\n', ' ', chunk.page_content)

# further split to avoid long chunks
chunks = char_splitter.split_documents(md_chunks)

In [10]:
print(len(chunks))
print(chunks[0].metadata)
print(chunks[1].metadata)

22
{'Document Title': 'Introduction to Data and Data Science', 'Topic Title': 'Analysis vs Analytics'}
{'Document Title': 'Introduction to Data and Data Science', 'Topic Title': 'Analysis vs Analytics'}


##### Recursive Character Text Splitter

> it recursively splits text into chunks using a hierarchy of separators, starting with `\n\n`

> if the chunk is still too large, it moves on to the next separator, and so on

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
rec_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100, separators=["\n\n", "\n", ".", " ", ""])

chunks = rec_splitter.split_documents(docs)

print(len(chunks))

24


In [13]:
print(chunks[0].page_content)
print(chunks[1].page_content)

# Introduction to Data and Data Science

## Analysis vs Analytics
Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis


In [14]:
print(chunks[9].page_content)
print(chunks[10].page_content)

## Programming Languages & Software Employed in Data Science - All the Tools You Need
Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execute the same action
